# Full-slice GP and CNN reconstruction of raw dMRI

Both methods receive raw diffusion-weighted intensities. Each complete slice is divided into overlapping patches, reconstructed locally, and joined by weighted overlap-add blending.

## 1. Prepare the Colab environment

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path('/content/dMRI_Interpolation_Repository')
DATA_ROOT = Path('/content/dMRI_Data')
if not PROJECT_ROOT.is_dir():
    from google.colab import files
    print('Upload dMRI_Interpolation_Repository.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project directory not found: {PROJECT_ROOT}')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    from google.colab import files
    print('Upload dMRI_Data.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if not archives:
        raise FileNotFoundError('No dMRI data ZIP archive was uploaded.')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall('/content')
if not (DATA_ROOT / 'TENSOR_2x2x2.nii.gz').is_file():
    raise FileNotFoundError(f'dMRI data folder not found: {DATA_ROOT}')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(PROJECT_ROOT / 'requirements_colab.txt')
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project ready:', PROJECT_ROOT)
print('External dMRI data ready:', DATA_ROOT)

## 2. Load one complete diffusion-weighted slice

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dmri_io import load_dmri_study
from dmri_gp import patch_gp_upsample

study = load_dmri_study(PROJECT_ROOT)
SLICE_INDEX, GRADIENT_INDEX = 19, 15
FACTOR = 2
PATCH_SIZE = 16
PATCH_STRIDE = 14
reference = np.asarray(
    study.data[:, :, SLICE_INDEX, GRADIENT_INDEX], dtype=np.float32
)
low = reference[::FACTOR, ::FACTOR]
print('Reference:', reference.shape, 'Low resolution:', low.shape)

## 3. Reconstruct and join all GP patches

In [ ]:
gp = patch_gp_upsample(
    low, reference.shape, patch_size=PATCH_SIZE, factor=FACTOR,
    stride=PATCH_STRIDE, blending='hann', preserve_samples=True,
    optimize_kernel=False, show_progress=True, return_details=True,
)
gp_slice = gp['reconstructed']
mask = np.ones(reference.shape, dtype=bool)
mask[::FACTOR, ::FACTOR] = False
gp_rmse = float(np.sqrt(np.mean((gp_slice[mask]-reference[mask])**2)))
print('Joined GP patches:', gp['patch_count'])
print('GP RMSE at unsampled pixels:', gp_rmse)

## 4. Reconstruct and join all CNN patches

In [ ]:
from dmri_cnn import cnn_reconstruct_slice

CNN_EPOCHS_PER_PATCH = 5  # Increase for the long configuration.
cnn = cnn_reconstruct_slice(
    reference, patch_size=PATCH_SIZE, factor=FACTOR,
    stride=PATCH_STRIDE, blending='hann', preserve_samples=True,
    epochs_per_patch=CNN_EPOCHS_PER_PATCH, show_progress=True,
)
cnn_slice = cnn['reconstructed']
print('Joined CNN patches:', cnn['patch_count'])
print('CNN RMSE at unsampled pixels:', cnn['rmse_unsampled'])

## 5. Compare the complete reconstructed slices

In [ ]:
images = (
    reference,
    low,
    gp_slice,
    cnn_slice,
    np.abs(gp_slice-reference),
    np.abs(cnn_slice-reference),
)
titles = (
    'Reference', 'Low resolution', 'GP full slice', 'CNN full slice',
    'GP absolute error', 'CNN absolute error',
)
cmaps = ('gray', 'gray', 'gray', 'gray', 'magma', 'magma')
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, image, title, cmap in zip(axes.ravel(), images, titles, cmaps):
    shown = ax.imshow(image.T, cmap=cmap, origin='lower')
    ax.set_title(title)
    ax.axis('off')
    if 'error' in title.lower():
        fig.colorbar(shown, ax=ax, fraction=0.046)
plt.tight_layout()

output_dir = Path('/content/results/full_slice_reconstruction')
output_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    output_dir / 'gp_cnn_complete_slice.npz',
    reference=reference, low_resolution=low,
    gp=gp_slice, cnn=cnn_slice,
    gp_absolute_error=np.abs(gp_slice-reference),
    cnn_absolute_error=np.abs(cnn_slice-reference),
)
fig.savefig(output_dir / 'gp_cnn_complete_slice.png', dpi=200)
print('Saved:', output_dir)